[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/fastapi-certified/notebooks/day-06-crud-in-memory.ipynb#scrollTo=aa11bb22)

---
# Day 6 · CRUD API with In-Memory Storage
**certified-journeys / fastapi-certified** · Practice · Day 6 of FastAPI for Python Engineers

> **Goal for today:** Build a complete, production-shaped Items API from scratch — Pydantic model layers, all five CRUD operations with correct status codes, pagination, and refactoring into a reusable `APIRouter`.


In [ ]:
%pip install -q fastapi httpx


## Step 1 · Three-Layer Pydantic Models — The Foundation

A production CRUD API separates its Pydantic models into three layers:

| Layer | Purpose | Fields |
|---|---|---|
| `ItemBase` | Shared fields — validation rules live here | `name`, `description`, `price` |
| `ItemCreate` | Input model — extends base; adds write-only fields | inherits `ItemBase` |
| `ItemOut` | Output model — extends base; adds server-generated fields | `id`, plus `ItemBase` |

This prevents accidental field leakage (e.g., internal cost margin, admin notes) and makes the API contract explicit. The internal storage dict can hold any fields; `ItemOut` controls what leaves the API.


In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from typing import Optional

# --- Layer 1: Base — shared validation lives here ---
class ItemBase(BaseModel):
    name: str = Field(..., min_length=1, max_length=100)
    description: Optional[str] = Field(None, max_length=500)
    price: float = Field(..., gt=0)  # must be > 0

# --- Layer 2: Create — what clients send on POST ---
class ItemCreate(ItemBase):
    # In a real app, you might add: category_id, supplier_id, etc.
    # For now, inherits everything from ItemBase
    pass

# --- Layer 3: Out — what clients receive ---
class ItemOut(ItemBase):
    id: int            # server-generated
    # internal_cost_margin: float  ← NOT here — stays server-side only

# Test the models independently before wiring into routes
item = ItemBase(name="Hammer", price=9.99)
print("Base model:", item)

# Validation: price must be > 0
try:
    bad = ItemBase(name="Free", price=-1)
except Exception as e:
    print("Validation error (expected):", type(e).__name__)

created = ItemCreate(name="Wrench", description="12mm", price=14.50)
print("Create model:", created)

out = ItemOut(id=1, name="Wrench", description="12mm", price=14.50)
print("Out model:", out)


**What just happened?**

- `ItemBase` holds all validation rules in one place — no duplication across `ItemCreate` and `ItemOut`.
- **`price=-1` fails validation** at model construction time — not at the route handler.
- `ItemOut` adds `id` (server-generated) without exposing it as a client-writable field.
- This three-layer pattern scales to any CRUD resource: swap `Item` for `Product`, `User`, `Order`, etc.


## Step 2 · POST /items — Create with 201 and Conflict Detection

The create route:
- Accepts `ItemCreate` in the request body
- Returns `ItemOut` (filters to public fields only)
- Returns **201 Created** on success
- Returns **409 Conflict** if the client tries to re-use an existing ID

The in-memory store is a simple `dict[int, dict]`. In production, replace this with SQLAlchemy + a real DB — the route logic stays the same.


In [ ]:
from fastapi import FastAPI, HTTPException, status

app = FastAPI()

# In-memory store: id → item dict (includes internal fields not in ItemOut)
items_db: dict[int, dict] = {}
next_id: int = 1

@app.post(
    "/items",
    response_model=ItemOut,
    status_code=status.HTTP_201_CREATED,
)
def create_item(item: ItemCreate):
    global next_id
    new_item = {
        "id": next_id,
        **item.dict(),
        "internal_cost_margin": item.price * 0.3,  # internal — not in ItemOut
    }
    items_db[next_id] = new_item
    next_id += 1
    return new_item  # FastAPI filters through ItemOut

client = TestClient(app)

# Create first item
r1 = client.post("/items", json={"name": "Hammer", "price": 9.99})
print("Create 1:", r1.status_code, r1.json())  # 201, id=1, no internal_cost_margin

# Create second item with description
r2 = client.post("/items", json={"name": "Wrench", "description": "12mm", "price": 14.50})
print("Create 2:", r2.status_code, r2.json())  # 201, id=2

# Confirm internal_cost_margin is NOT in the response
print("Has internal field:", "internal_cost_margin" in r1.json())  # False
print("In-memory store   :", list(items_db.keys()))  # [1, 2]


**What just happened?**

- `response_model=ItemOut` silently strips `internal_cost_margin` from the response — it exists in the store but never leaves the API.
- `status_code=status.HTTP_201_CREATED` signals to clients that a resource was created (not just "OK").
- The store uses a simple `dict` — for production, swap with SQLAlchemy without changing the route logic.
- `next_id` acts as an auto-increment counter — in production, the DB handles this.


## Step 3 · GET /items — List with Pagination

List endpoints must always be paginated — returning all records is dangerous at scale. FastAPI makes this clean with query parameters that have defaults and validation:

- `skip: int = 0` — offset from the start
- `limit: int = 20` — maximum records to return

Return the total count alongside the page so clients can calculate remaining pages without a second request.


In [ ]:
from typing import List

@app.get("/items", response_model=List[ItemOut])
def list_items(
    skip: int = 0,
    limit: int = 20,
    name_filter: Optional[str] = None,   # optional search by name substring
):
    all_items = list(items_db.values())

    # Optional substring filter
    if name_filter:
        all_items = [i for i in all_items if name_filter.lower() in i["name"].lower()]

    # Apply pagination
    paginated = all_items[skip: skip + limit]
    return paginated  # FastAPI maps each dict through ItemOut

# --- Test pagination ---

# Seed more items for pagination demo
for name, price in [("Screwdriver", 7.50), ("Pliers", 12.00), ("Tape Measure", 18.99)]:
    client.post("/items", json={"name": name, "price": price})

# Get all (up to limit=20)
all_resp = client.get("/items")
print("Total items:", len(all_resp.json()))  # 5

# Page 1: skip=0, limit=2
p1 = client.get("/items", params={"skip": 0, "limit": 2})
print("Page 1:", [i["name"] for i in p1.json()])  # first 2

# Page 2: skip=2, limit=2
p2 = client.get("/items", params={"skip": 2, "limit": 2})
print("Page 2:", [i["name"] for i in p2.json()])  # next 2

# Filter by name
filtered = client.get("/items", params={"name_filter": "wr"})
print("Filtered 'wr':", [i["name"] for i in filtered.json()])  # Wrench


**What just happened?**

- `skip` and `limit` are plain query parameters — FastAPI reads them from the URL and applies Pydantic defaults.
- **`response_model=List[ItemOut]`** maps every dict in the list through `ItemOut` before sending.
- The optional `name_filter` parameter shows how to add search without a complex query layer.
- In production, push `skip`/`limit` and filtering into the SQL query (e.g., `.offset().limit().filter()`) — never load all rows into Python memory.


## Step 4 · GET /items/{id} and PUT /items/{id} — Read and Update

Single-item read and update routes both need the same 404 guard: if the item doesn't exist, raise `HTTPException(404)` immediately.

For `PUT`, accept a full `ItemCreate` body (all fields required). For partial updates, you'd use `PATCH` with an optional-field model — we'll keep `PUT` here for clarity.


In [ ]:
from fastapi import HTTPException

def _get_or_404(item_id: int) -> dict:
    """Helper: return item or raise 404. DRY — used by GET, PUT, DELETE."""
    if item_id not in items_db:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail=f"Item {item_id} not found"
        )
    return items_db[item_id]

@app.get("/items/{item_id}", response_model=ItemOut)
def get_item(item_id: int):
    return _get_or_404(item_id)

@app.put("/items/{item_id}", response_model=ItemOut)
def update_item(item_id: int, item: ItemCreate):
    existing = _get_or_404(item_id)  # raises 404 if missing
    # Preserve internal fields, update only ItemCreate fields
    updated = {**existing, **item.dict()}
    items_db[item_id] = updated
    return updated

# --- Test GET ---
r_get = client.get("/items/1")
print("GET  item 1:", r_get.status_code, r_get.json())

r_miss = client.get("/items/999")
print("GET  missing:", r_miss.status_code, r_miss.json())

# --- Test PUT ---
r_put = client.put("/items/1", json={"name": "Big Hammer", "price": 19.99})
print("PUT  item 1:", r_put.status_code, r_put.json())

r_put_miss = client.put("/items/999", json={"name": "Ghost", "price": 1.00})
print("PUT  missing:", r_put_miss.status_code, r_put_miss.json())


**What just happened?**

- **`_get_or_404` is a plain helper** — not a dependency or a route. It centralizes the 404 check so GET, PUT, and DELETE all share identical guard logic.
- `PUT` preserves `internal_cost_margin` by merging into `existing` before overwriting.
- `response_model=ItemOut` ensures the updated record is filtered before returning — internal fields stay internal.
- **Returning 200 for PUT is correct** — the resource was found and updated. Use 201 only for creation.


## Step 5 · DELETE /items/{id} — Delete with 204 No Content

`DELETE` should return **204 No Content** on success. This tells the client "it's gone — there's nothing to show you." Returning a body on 204 is a protocol error — FastAPI suppresses it automatically when `None` is returned.

Also handle the 409 Conflict case: if two requests try to create with the same ID simultaneously, the second gets a 409. We'll demonstrate this pattern explicitly.


In [ ]:
@app.delete("/items/{item_id}", status_code=status.HTTP_204_NO_CONTENT)
def delete_item(item_id: int):
    _get_or_404(item_id)  # raises 404 if not found — ensures we don't silently ignore bad IDs
    del items_db[item_id]
    # Return None — FastAPI correctly sends no body with 204

# --- Test DELETE ---
r_del = client.delete("/items/2")
print("DELETE item 2:", r_del.status_code)  # 204
print("Body (should be empty):", r_del.content)  # b''

# Confirm item 2 is gone
r_gone = client.get("/items/2")
print("GET after DELETE:", r_gone.status_code)  # 404

# DELETE non-existent
r_del_miss = client.delete("/items/999")
print("DELETE missing:", r_del_miss.status_code)  # 404

# --- Demonstrate 409 Conflict pattern (duplicate detection) ---
# Simulate a POST-with-id-conflict scenario using a separate endpoint

@app.post("/items/reserve/{item_id}", status_code=status.HTTP_201_CREATED)
def reserve_id(item_id: int, item: ItemCreate):
    """POST to a specific ID — useful in APIs that allow client-controlled IDs."""
    if item_id in items_db:
        raise HTTPException(
            status_code=status.HTTP_409_CONFLICT,
            detail=f"Item with id={item_id} already exists"
        )
    items_db[item_id] = {"id": item_id, **item.dict()}
    return items_db[item_id]

# First reservation → 201
r_res1 = client.post("/items/reserve/100", json={"name": "Reserved", "price": 1.00})
print("Reserve 100:", r_res1.status_code)

# Duplicate reservation → 409
r_res2 = client.post("/items/reserve/100", json={"name": "Duplicate", "price": 2.00})
print("Reserve 100 again:", r_res2.status_code, r_res2.json())


**What just happened?**

- **204 returns no body** — `r_del.content` is `b''`.
- Raising 404 on DELETE for a non-existent resource is the correct behavior — a silent 204 would lie to the client.
- **409 Conflict** is the right code when a POST would violate uniqueness — not 400 (bad request) which implies a malformed payload.
- The full status code map: 201 Create, 200 Read/Update, 204 Delete, 404 Not Found, 409 Conflict.


## Step 6 · Refactor into an `APIRouter`

As APIs grow, keeping all routes in `main.py` becomes unmanageable. FastAPI's `APIRouter` lets you group related routes into their own module and mount them on the main app.

```
routers/
  items.py     ← APIRouter for /items
  users.py     ← APIRouter for /users
main.py        ← FastAPI app + include_router()
```

In a notebook we do this in cells instead of files — the pattern is identical.


In [ ]:
from fastapi import FastAPI, APIRouter, HTTPException, status
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from typing import Optional, List

# ── Pydantic models (same three-layer pattern) ──────────────────────────────
class ItemBase(BaseModel):
    name: str = Field(..., min_length=1, max_length=100)
    description: Optional[str] = None
    price: float = Field(..., gt=0)

class ItemCreate(ItemBase):
    pass

class ItemOut(ItemBase):
    id: int

# ── In-memory store ──────────────────────────────────────────────────────────
store: dict[int, dict] = {}
counter = 1

def _find_or_404(item_id: int) -> dict:
    if item_id not in store:
        raise HTTPException(status_code=404, detail=f"Item {item_id} not found")
    return store[item_id]

# ── Router: groups all /items routes ────────────────────────────────────────
# prefix means every route defined here is under /items automatically
items_router = APIRouter(prefix="/items", tags=["items"])

@items_router.post("", response_model=ItemOut, status_code=status.HTTP_201_CREATED)
def create_item(item: ItemCreate):
    global counter
    record = {"id": counter, **item.dict()}
    store[counter] = record
    counter += 1
    return record

@items_router.get("", response_model=List[ItemOut])
def list_items(skip: int = 0, limit: int = 20):
    return list(store.values())[skip: skip + limit]

@items_router.get("/{item_id}", response_model=ItemOut)
def get_item(item_id: int):
    return _find_or_404(item_id)

@items_router.put("/{item_id}", response_model=ItemOut)
def update_item(item_id: int, item: ItemCreate):
    existing = _find_or_404(item_id)
    store[item_id] = {**existing, **item.dict()}
    return store[item_id]

@items_router.delete("/{item_id}", status_code=status.HTTP_204_NO_CONTENT)
def delete_item(item_id: int):
    _find_or_404(item_id)
    del store[item_id]

# ── Main app: mount the router ───────────────────────────────────────────────
main_app = FastAPI(title="Items API v1")
main_app.include_router(items_router)  # all /items routes are now registered

# Optional: a health check at app level
@main_app.get("/health")
def health():
    return {"status": "ok", "items": len(store)}

# ── Verify with TestClient ───────────────────────────────────────────────────
client_main = TestClient(main_app)

r_create = client_main.post("/items", json={"name": "Widget", "price": 4.99})
print("POST /items:", r_create.status_code, r_create.json())

r_list = client_main.get("/items")
print("GET  /items:", r_list.status_code, r_list.json())

r_get = client_main.get("/items/1")
print("GET  /items/1:", r_get.status_code, r_get.json())

r_health = client_main.get("/health")
print("GET  /health:", r_health.json())


**What just happened?**

- `APIRouter(prefix="/items")` scopes all route paths to `/items` — no prefix repetition in each `@router.get(...)` call.
- **`tags=["items"]`** groups these routes in the OpenAPI docs under an "items" section.
- `main_app.include_router(items_router)` is the single line that mounts the entire router.
- The `main_app` stays clean — it only includes the router and its own routes (like `/health`).


## Step 7 · Verifying the Full CRUD Contract

A good CRUD test suite exercises every route and every error path. Let's write a systematic verification across all five operations and confirm the correct status codes throughout.


In [ ]:
# Fresh state for a clean test run
store.clear()
counter = 1

def assert_status(resp, expected: int, label: str):
    icon = "✓" if resp.status_code == expected else "✗"
    body = resp.json() if resp.content else "(no body)"
    print(f"{icon} [{label}] expected={expected} got={resp.status_code}  body={body}")

# CREATE two items
r1 = client_main.post("/items", json={"name": "Hammer", "price": 9.99})
r2 = client_main.post("/items", json={"name": "Wrench", "description": "12mm", "price": 14.50})
assert_status(r1, 201, "POST item 1")
assert_status(r2, 201, "POST item 2")

# LIST with pagination
r_list_all  = client_main.get("/items")
r_list_page = client_main.get("/items", params={"skip": 1, "limit": 1})
assert_status(r_list_all,  200, "GET all items")
assert_status(r_list_page, 200, "GET page 2")
print(f"  Page 2 items: {[i['name'] for i in r_list_page.json()]}")

# READ
r_get_ok   = client_main.get("/items/1")
r_get_miss = client_main.get("/items/999")
assert_status(r_get_ok,   200, "GET item 1")
assert_status(r_get_miss, 404, "GET missing")

# UPDATE
r_put_ok   = client_main.put("/items/1", json={"name": "Big Hammer", "price": 19.99})
r_put_miss = client_main.put("/items/999", json={"name": "Ghost", "price": 1.00})
assert_status(r_put_ok,   200, "PUT item 1")
assert_status(r_put_miss, 404, "PUT missing")
print(f"  Updated name: {r_put_ok.json()['name']}")

# DELETE
r_del_ok   = client_main.delete("/items/2")
r_del_miss = client_main.delete("/items/999")
assert_status(r_del_ok,   204, "DELETE item 2")
assert_status(r_del_miss, 404, "DELETE missing")

# Confirm deletion
r_gone = client_main.get("/items/2")
assert_status(r_gone, 404, "GET deleted item")

# Validate field filtering — no internal fields leak
item_body = r_get_ok.json()
print(f"\nItemOut keys: {sorted(item_body.keys())}")
assert set(item_body.keys()) <= {"id", "name", "description", "price"}, "Unexpected fields in response!"
print("✓ No internal fields in response")


**What just happened?**

- Every route was exercised — both happy path and error path.
- **The status code contract is verified**: 201/200/204 for success, 404 for missing items.
- `assert set(item_body.keys()) <= {...}` ensures no internal field accidentally leaked into the response.
- This pattern translates directly to pytest test cases — replace `assert_status` with `assert` and you have a production test suite.


## Step 8 · Inspecting the OpenAPI Schema from the Router

When you use `APIRouter`, FastAPI merges the router's routes into the app's OpenAPI schema. This means you get full, correct documentation for free — including the `tags` grouping.


In [ ]:
import json

schema = client_main.get("/openapi.json").json()

# List all registered paths
print("Registered paths:")
for path, methods in schema["paths"].items():
    for method in methods:
        route_info = methods[method]
        tags = route_info.get("tags", [])
        print(f"  {method.upper():<8} {path:<25} tags={tags}")

# Confirm ItemOut schema in components
item_out_props = schema["components"]["schemas"]["ItemOut"]["properties"]
print("\nItemOut schema properties:", list(item_out_props.keys()))

# Confirm POST /items returns 201 (not 200)
post_response_codes = list(schema["paths"]["/items"]["post"]["responses"].keys())
print("POST /items response codes:", post_response_codes)  # ['201']


**What just happened?**

- `include_router` fully merged the router's routes into the app's OpenAPI schema.
- **Tags appear in the schema** — OpenAPI clients and the Swagger UI at `/docs` will group items routes under "items".
- The schema confirms `POST /items` returns **201, not 200** — your API contract matches the implementation.
- `ItemOut` properties are listed in the schema — clients can generate typed SDKs directly from this.


In [ ]:
# Challenge: Extend the Items API
#
# Add these features to the existing items_router (or build a fresh one):
#
# 1. ItemUpdate model: all fields optional (for PATCH semantics)
#    class ItemUpdate(BaseModel):
#        name: Optional[str] = None
#        description: Optional[str] = None
#        price: Optional[float] = None
#
# 2. PATCH /items/{id} → 200
#    Only update fields that are present in the request body
#    (hint: use item.dict(exclude_unset=True) to get only set fields)
#    Raise 404 if item not found
#
# 3. GET /items with a min_price filter query param
#    Only return items with price >= min_price
#
# 4. A second router for /categories:
#    POST /categories → create category {id, name}
#    GET /categories  → list all categories
#    Mount both routers on a fresh FastAPI app
#
# Test with TestClient:
#   - PATCH with partial update preserves unchanged fields
#   - min_price filter returns only matching items
#   - Both /items and /categories routes work on the same app

from fastapi import FastAPI, APIRouter, HTTPException, status
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from typing import Optional, List

# Your solution here
# class ItemUpdate(BaseModel): ...
# items_router2 = APIRouter(prefix="/items", tags=["items"])
# categories_router = APIRouter(prefix="/categories", tags=["categories"])
# @items_router2.patch("/{item_id}", ...) ...
# @items_router2.get("", ...) with min_price filter ...
# @categories_router.post(...) ...
# @categories_router.get(...) ...
# app = FastAPI()
# app.include_router(items_router2)
# app.include_router(categories_router)
# client = TestClient(app)
# ...


---
## Day 6 key concepts recap

| Concept | What to remember |
|---|---|
| Three-layer models | `ItemBase` → `ItemCreate` → `ItemOut`; prevents field leakage, avoids duplication |
| `dict` as store | Simple in-memory CRUD; swap for SQLAlchemy without changing route logic |
| 201 / 200 / 204 / 404 / 409 | Create / Read+Update / Delete / Not Found / Conflict |
| Pagination | `skip` + `limit` query params; always cap `limit` to a sane maximum |
| `APIRouter(prefix=...)` | Groups routes by resource; cleanly mountable with `include_router()` |
| `tags=` | Groups routes in OpenAPI docs / Swagger UI |
| `item.dict(exclude_unset=True)` | Key for PATCH — only updates explicitly-provided fields |

> **Tip:** Design your Pydantic models in three layers: `ItemBase`, `ItemCreate`, `ItemOut`. This prevents accidental field leakage.

---
## What's next
**Day 7** → Connecting a real database — wire your `APIRouter` to SQLAlchemy, Alembic migrations, and async sessions.

Mark Day 6 complete in your [tracker](../index.html).
